# W7D2 — CLIP and Vision-Language Models — Lab

**Week 7 · Day 2 · Vision, language and speech** · Lab

<a href="https://colab.research.google.com/github/0xRush/AIEP_Olo_student/blob/master/week_7_multimodal/labs/D2_clip_vlm/D2_clip_vlm_blank.ipynb" target="_blank">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **CLIP** turns a photo and a sentence into vectors in **the same space**.
> If the sentence describes the photo, the two vectors point the same way.
> A **VLM** (vision-language model) goes one step further: you **ask** the photo a question
> and it **writes** the answer.

CLIP is a **judge**: give it a photo and some sentences, and it gives every sentence a score —
but it cannot write a single word. A VLM is a **describer**: it looks at the photo and talks about it.

<div dir="rtl" align="right">

# W7D2 — CLIP ونماذج الرؤية واللغة

**الأسبوع السابع · اليوم الثاني · الرؤية واللغة والكلام** · معمل

> نموذج **CLIP** يحول الصورة والجملة إلى متجهات في **فضاء واحد**.
> إذا كانت الجملة تصف الصورة، يشير المتجهان إلى الاتجاه نفسه.
> أما **نموذج الرؤية واللغة** فيذهب خطوة أبعد: تسأل الصورة سؤالا، فيكتب لك الجواب.

نموذج CLIP مثل **حكم**: تعطيه صورة وعدة جمل، فيعطي كل جملة درجة، لكنه لا يكتب كلمة واحدة.
ونموذج الرؤية واللغة مثل **واصف**: ينظر إلى الصورة ويتكلم عنها.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

### The whole idea in one picture

```
  CLIP — scores a sentence you wrote                                   (Section 2)

     photo ───► image encoder ───► 512 numbers ──┐
                                                 ├──► cosine = how well they match
     text  ───► text encoder  ───► 512 numbers ──┘

  VLM — writes the answer itself                                       (Section 3)

     photo    ───► vision encoder ───► connector ───► image tokens ──┐
                                                                     ├──► language model ───► answer
     question ─────────────────────────────────────► text tokens ────┘
```

### Today's pipeline

| step | what you do | what you get |
|---|---|---|
| Setup | load the data and three helpers | 200 photo–caption pairs, 400 class photos |
| 1 · Warm-up | a toy example, then real CLIP on one photo | the idea, by eye |
| 2.1 · Embed | every photo and caption → a vector | `SIMILARITY`, a 200 × 200 table |
| 2.2 · Search | read a column, read a row | text → image, image → text |
| 2.3 · Measure | find the rank of the right answer | Recall@1, Recall@5, MRR |
| 2.4 · Zero-shot | class name → sentence → vector | accuracy with **0** labels |
| 2.5 · Break it | 8 hard captions | where CLIP fails |
| 3.1 · Open the VLM | count its parts and its image tokens | 3 parts, 64 vs 832 tokens |
| 3.2 · Ask | finish `ask(image, question)` | captions for 3 photos |
| 3.3 · Re-test | CLIP's failures, low vs high detail | right answers and seconds |
| Checks | run the last cell | 6 green checks |

<div dir="rtl" align="right">

### الفكرة كلها في صورة

**نموذج CLIP (القسم ٢):** الصورة ← مرمز الصور ← ٥١٢ رقما. والجملة ← مرمز النصوص ← ٥١٢ رقما.
ثم جيب التمام بينهما = درجة التطابق.

**نموذج الرؤية واللغة (القسم ٣):** الصورة ← مرمز الرؤية ← الموصل ← رموز صورة.
ثم رموز الصورة مع سؤالك ← النموذج اللغوي ← الجواب.

### خطة اليوم

1. **الإحماء:** مثال صغير، ثم CLIP الحقيقي على صورة واحدة.
2. **نموذج CLIP:** حول كل صورة وكل تعليق إلى متجه، وابحث في الاتجاهين، وقس جودة البحث، وصنف بلا تدريب، ثم اكسره بثمانية تعليقات صعبة.
3. **نموذج الرؤية واللغة:** افتحه وعد أجزاءه ورموزه، واسأله، ثم أعد عليه الأسئلة التي فشل فيها CLIP.
4. **الفحوص:** شغل الخلية الأخيرة.

</div>

## Models and data

| model | what it is | size |
|---|---|---|
| `openai/clip-vit-base-patch32` | **CLIP**: an image encoder + a text encoder, trained together on 400M photo–caption pairs | 605 MB |
| `HuggingFaceTB/SmolVLM-500M-Instruct` | a small **VLM**: vision encoder + connector + language model | 1.0 GB |

- `image_captions_small` — 200 photos, each with its **one** right caption, so we can score the
  search. The last 8 captions are **hard cases**: counting, left/right, a word written in the photo,
  and one caption in Arabic.
- `small_image_5class` — yesterday's 400 photos in 5 classes, for zero-shot classification.

The first run downloads about 1.6 GB. The VLM is the slow part: about 1.5 s per answer at low
detail and 11 s at high detail on a laptop CPU.

<div dir="rtl" align="right">

## النماذج والبيانات

- نموذج **CLIP**: مرمز صور ومرمز نصوص، دربا معا على ٤٠٠ مليون زوج من صورة وتعليق. حجمه ٦٠٥ ميجابايت.
- نموذج **SmolVLM**: نموذج رؤية ولغة صغير، فيه مرمز رؤية وموصل ونموذج لغوي. حجمه ١ جيجابايت.
- بيانات **image_captions_small**: ٢٠٠ صورة، لكل صورة تعليق صحيح **واحد**، لذلك نستطيع تقييم البحث. والتعليقات الثمانية الأخيرة **حالات صعبة**: العد، واليمين واليسار، وكلمة مكتوبة في الصورة، وتعليق باللغة العربية.
- بيانات **small_image_5class**: صور الأمس الأربعمئة في خمس فئات، للتصنيف بلا تدريب.

التشغيل الأول ينزل نحو ١٫٦ جيجابايت. والجزء البطيء هو نموذج الرؤية واللغة: نحو ثانية ونصف لكل جواب بتفاصيل قليلة، ونحو ١١ ثانية بتفاصيل كثيرة.

</div>

## Setup

Run this first. On Colab it installs the course package; on your laptop it uses your copy of the
repo. It loads the data and defines three helpers you will just *use*:

- `load_image(path)` → the photo, ready for a model;
- `normalise(vectors)` → every row made length 1, so `a @ b` is a cosine;
- `show_images(images, titles)` → photos side by side.

<div dir="rtl" align="right">

## الإعداد

شغل هذه الخلية أولا. تحمل البيانات وتعرف ثلاث أدوات ستستعملها فقط:
فتح صورة، وجعل طول كل متجه يساوي ١ (فيصير الضرب النقطي جيب تمام)، وعرض الصور جنبا إلى جنب.

</div>

In [ ]:
# === Setup: data and helpers ========================================================
try:
    import aiep
except ImportError:                       # not installed: use the repo copy, or install it on Colab
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, device, versions
ensure("transformers", "torch", "matplotlib", "pandas")
seed_everything(42)

from aiep.data import get_dataset_dir, load_artefact
from aiep.checks import check, report

import json, textwrap, time, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

torch.set_grad_enabled(False)             # nothing is trained today
np.set_printoptions(precision=3, suppress=True)
warnings.filterwarnings("ignore", message=".*encountered in matmul")   # harmless NumPy noise on Macs

# --- the data --------------------------------------------------------------------------
CAPTION_ROOT = get_dataset_dir("image_captions_small")
CAPTIONS = pd.read_csv(CAPTION_ROOT / "captions.csv")                  # columns: file, caption, hard_case
CAPTION_IMAGES = [CAPTION_ROOT / "images" / name for name in CAPTIONS.file]

CLASS_PATHS = sorted((get_dataset_dir("small_image_5class") / "images").rglob("*.jpg"))
CLASS_LABELS = np.array([p.parent.name for p in CLASS_PATHS])          # the folder name is the class
CLASSES = sorted(set(CLASS_LABELS))

PROBE = json.loads(load_artefact("probe_results.json").read_text(encoding="utf-8"))   # yesterday's results


# --- three helpers ---------------------------------------------------------------------
def load_image(path):
    """Open a photo as RGB."""
    return Image.open(path).convert("RGB")


def normalise(vectors):
    """Make every row length 1, so a dot product is a cosine."""
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


def show_images(images, titles, suptitle=""):
    """Draw photos (paths or PIL images) side by side, each with a title."""
    fig, axes = plt.subplots(1, len(images), figsize=(2.7 * len(images), 3), squeeze=False)
    for ax, image, title in zip(axes[0], images, titles):
        ax.imshow(image if isinstance(image, Image.Image) else load_image(image))
        ax.set_title(title, fontsize=8)
        ax.axis("off")
    fig.suptitle(suptitle)
    plt.tight_layout()
    plt.show()


print(f"{len(CAPTIONS)} photo-caption pairs, {CAPTIONS.hard_case.notna().sum()} of them hard cases")
print(f"{len(CLASS_PATHS)} class photos: {CLASSES}")
print(f"yesterday's DINOv2 probe: {PROBE['dino_probe']:.4f} with 320 labels")
print(versions(), "| device:", device())

## Section 1 — Warm-up: one space for photos and words  *(~10 min)*

Nothing to write. Two demos:

1. **A toy.** Two photos and three captions, each a vector of only **2** numbers. This cosine table
   is everything CLIP does — real CLIP just uses 512 numbers.
2. **Real CLIP.** One photo against five sentences.

<div dir="rtl" align="right">

## القسم ١ — إحماء: فضاء واحد للصور والكلمات  *(نحو ١٠ دقائق)*

لا شيء تكتبه. عرضان:

1. **مثال صغير:** صورتان وثلاثة تعليقات، كل منها متجه من رقمين فقط. جدول جيب التمام هذا هو كل ما يفعله CLIP، لكن CLIP الحقيقي يستعمل ٥١٢ رقما.
2. **نموذج CLIP الحقيقي:** صورة واحدة مقابل خمس جمل.

</div>

In [ ]:
# === Demo 1: the toy — 2 numbers instead of 512 =======================================
TOY_PHOTOS = np.array([[0.9, 0.1],        # photo 1: a cat
                       [0.1, 0.9]])       # photo 2: a car
TOY_CAPTIONS = np.array([[0.8, 0.2],      # "a cat"
                         [0.2, 0.8],      # "a car"
                         [0.5, 0.5]])     # "a thing"

TOY = normalise(TOY_PHOTOS) @ normalise(TOY_CAPTIONS).T      # cosine of every photo with every caption
print(pd.DataFrame(TOY.round(3), index=["photo: cat", "photo: car"], columns=["a cat", "a car", "a thing"]))
print("\nThe best caption for a photo is the biggest number in its row. That is zero-shot classification.")
print('"a thing" gets the same score with both photos: true of everything, useful for nothing.')

In [ ]:
# === Demo 2: real CLIP scores one photo against five sentences ========================
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")   # resizes photos, splits text into tokens

photo = load_image(CAPTION_IMAGES[0])
sentences = ["a photo of a dog",
             "a photo of a pizza",
             "a photo of a street",
             CAPTIONS.caption[0],                      # the photo's real caption
             "a diagram of a neural network"]

inputs = clip_processor(text=sentences, images=photo, return_tensors="pt", padding=True)
scores = clip(**inputs).logits_per_image.softmax(-1)[0]      # one score per sentence; they add up to 1

show_images([photo], ["photo 0"])
for score, sentence in sorted(zip(scores.tolist(), sentences), reverse=True):
    print(f"{score:.3f}  {sentence}")

## Section 2 — CLIP: embed, search, measure, classify, break  *(~55 min)*

```
  200 photos   ──► embed_images ──► IMAGE_VECTORS (200, 512) ──┐
                                                               ├──► SIMILARITY (200 × 200)
  200 captions ──► embed_texts  ──► TEXT_VECTORS  (200, 512) ──┘          │
                                                                          ▼
                                       Task 2 search · Task 3 metrics · Task 5 hard cases
```

Five tasks. Build the table once in Task 1 — every task after it just **reads** it.

<div dir="rtl" align="right">

## القسم ٢ — نموذج CLIP: ضمن، وابحث، وقس، وصنف، واكسر  *(نحو ٥٥ دقيقة)*

خمس مهام. تبني جدول التشابه مرة واحدة في المهمة ١، وكل المهام بعدها **تقرأ** منه فقط:
الصور ← متجهات الصور، والتعليقات ← متجهات التعليقات، ثم الجدول ← البحث والمقاييس والحالات الصعبة.

</div>

### Task 1 — Turn photos and captions into vectors

An **embedding** is a list of numbers that stands for a photo or a sentence. CLIP gives 512 numbers
for each, and puts both kinds in the same space.

The first cell below is **given**: `embed_images` and `embed_texts` turn photos and sentences into
vectors of length 1. Just run it and read it.

**Your job is one line:** build the table `SIMILARITY` — every photo vector times every caption
vector. Row *i* is photo *i* against all 200 captions.

✅ **You should see:** `(200, 200)`, and in the picture after it, a bright diagonal.

<div dir="rtl" align="right">

### المهمة ١ — حول الصور والتعليقات إلى متجهات

**التضمين** قائمة أرقام تمثل صورة أو جملة. يعطي CLIP لكل منهما ٥١٢ رقما، ويضع النوعين في فضاء واحد.

الخلية الأولى **جاهزة**: فيها دالتان تحولان الصور والجمل إلى متجهات طولها ١. شغلها واقرأها فقط.

**مهمتك سطر واحد:** ابن جدول التشابه، أي كل متجه صورة مضروبا في كل متجه تعليق.
الصف رقم i هو الصورة رقم i مقابل كل التعليقات.

✅ **سترى:** جدولا حجمه ٢٠٠ في ٢٠٠، وفي الرسم بعده قطرا لامعا.

</div>

In [ ]:
# === Given: turn photos and sentences into CLIP vectors =================================
def embed_images(paths):
    """One CLIP vector (length 1) per photo."""
    vectors = []
    for start in range(0, len(paths), 32):                      # 32 photos at a time
        photos = [load_image(p) for p in paths[start:start + 32]]
        pixels = clip_processor(images=photos, return_tensors="pt")["pixel_values"]
        vectors.append(clip.get_image_features(pixel_values=pixels).pooler_output.numpy())
    return normalise(np.concatenate(vectors))


def embed_texts(texts):
    """One CLIP vector (length 1) per sentence."""
    tokens = clip_processor(text=list(texts), return_tensors="pt", padding=True, truncation=True)
    return normalise(clip.get_text_features(**tokens).pooler_output.numpy())


IMAGE_VECTORS = embed_images(CAPTION_IMAGES)        # 200 photos   -> (200, 512)
TEXT_VECTORS = embed_texts(CAPTIONS.caption)        # 200 captions -> (200, 512)
print("photo vectors:  ", IMAGE_VECTORS.shape)
print("caption vectors:", TEXT_VECTORS.shape)

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) A @ B.T multiplies every row of A by every row of B.
# 2) SIMILARITY = IMAGE_VECTORS @ TEXT_VECTORS.T
#
# خطوة ١: الضرب في المنقول يضرب كل صف من الأول في كل صف من الثاني.
# خطوة ٢: اضرب متجهات الصور في منقول متجهات التعليقات.
# ────────────────────────────────────────────────────────────────────

# TODO: SIMILARITY = IMAGE_VECTORS @ ... (every photo vector times every caption vector)
# مهمة: اضرب متجهات الصور في منقول متجهات التعليقات.
print("similarity table:", SIMILARITY.shape, "<- should be (200, 200)")

In [ ]:
# === Look at the table: the top-left corner, 8 photos x their 8 captions ================
plt.figure(figsize=(5.5, 4.5))
plt.imshow(SIMILARITY[:8, :8], cmap="viridis")
plt.colorbar(label="cosine")
plt.xlabel("caption number"); plt.ylabel("photo number")
plt.title("bright diagonal = each photo matches its own caption")
plt.show()

### Task 2 — Search both ways

One table, read two ways:

```
                caption 0   caption 1   ...   caption 199
   photo 0   [    0.31        0.12      ...      0.08    ]  ◄── a ROW: one photo vs all captions
   photo 1   [    0.10        0.29      ...      0.15    ]      = image → text search
   ...                         ▲
                               └── a COLUMN: one caption vs all photos
                                   = text → image search
```

**Your job is one line:** finish `top_k(scores, k)` — the **positions** of the `k` biggest scores,
best first. Tip: `np.argsort` sorts from small to big, so sort `-scores` instead.

✅ **You should see:** `[1 2]`, then three rows of photos that fit the sentences, and photo 7's own
caption at the top of its list.

<div dir="rtl" align="right">

### المهمة ٢ — ابحث في الاتجاهين

جدول واحد يقرأ بطريقتين:

- **الصف** = صورة واحدة مقابل كل التعليقات، أي بحث من الصورة إلى النص.
- **العمود** = تعليق واحد مقابل كل الصور، أي بحث من النص إلى الصورة.

**مهمتك سطر واحد:** أكمل الدالة `top_k` التي تعيد **مواقع** أكبر الدرجات، الأفضل أولا.
تلميح: الترتيب يكون من الأصغر إلى الأكبر، لذلك رتب سالب الدرجات.

✅ **سترى:** النتيجة [1 2]، ثم ثلاثة صفوف من الصور تناسب الجمل، وتعليق الصورة ٧ في أول قائمتها.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) np.argsort(x) gives the positions that sort x from small to big.
# 2) np.argsort(-scores) is big to small; keep the first k with [:k].
#
# خطوة ١: دالة الترتيب في نمباي تعطي المواقع من الأصغر إلى الأكبر.
# خطوة ٢: رتب سالب الدرجات لتحصل على الأكبر أولا، ثم خذ أول k فقط.
# ────────────────────────────────────────────────────────────────────

def top_k(scores, k=5):
    """Positions of the k highest scores, best first."""
    # TODO: return np.argsort(...)[:k]  (sort -scores, so the biggest comes first)
    # مهمة: أعد مواقع أكبر k قيم في الدرجات، الأكبر أولا.


print(top_k(np.array([0.1, 0.9, 0.5]), 2), "<- should be [1 2]")

In [ ]:
# === Search with top_k ================================================================
# text -> image: a NEW sentence gives one new column of scores
for query in ["two dogs playing on the grass", "a plate of food on a table", "a train at a station"]:
    scores = IMAGE_VECTORS @ embed_texts([query])[0]         # one score per photo
    best = top_k(scores)
    show_images([CAPTION_IMAGES[i] for i in best], [f"score {scores[i]:.2f}" for i in best], f'"{query}"')

# image -> text: one ROW of the table
row = SIMILARITY[7]
show_images([CAPTION_IMAGES[7]], ["photo 7"])
print("the 3 best captions for photo 7:")
for i in top_k(row, 3):
    print(f"  {row[i]:.3f}  {CAPTIONS.caption[i]}" + ("   <- its own caption" if i == 7 else ""))

### Task 3 — How good is the search?

For every photo, find the **rank** of its own caption among all 200 (rank 1 = it came first).
`own_rank` does that for you. You turn the ranks into three numbers:

| metric | question it answers | perfect |
|---|---|---|
| **Recall@1** | how often is the right answer **1st**? | 1.0 |
| **Recall@5** | how often is it in the **top 5**? | 1.0 |
| **MRR** | the average of 1 / rank (1st → 1, 2nd → 0.5, 3rd → 0.33 …) | 1.0 |

Random guessing would give Recall@1 = 1/200 = 0.005.

**Your job is two lines:** `recall_5` is written for you as an example. Copy its pattern for
`recall_1`, then write `mrr`.

✅ **You should see:** image → text R@1 ≈ 0.57, R@5 ≈ 0.85, MRR ≈ 0.70 — about 100 times better than
guessing.

<div dir="rtl" align="right">

### المهمة ٣ — ما جودة البحث؟

لكل صورة، نجد **رتبة** تعليقها الصحيح بين التعليقات المئتين (الرتبة ١ = جاء أولا). الدالة `own_rank` تفعل هذا لك.
وأنت تحول الرتب إلى ثلاثة أرقام:

- **Recall@1**: كم مرة جاء الجواب الصحيح **أولا**؟
- **Recall@5**: كم مرة جاء ضمن **الخمسة الأوائل**؟
- **MRR**: متوسط واحد على الرتبة (الأول = ١، الثاني = ٠٫٥، الثالث = ٠٫٣٣).

التخمين العشوائي يعطي ٠٫٠٠٥ فقط.

**مهمتك سطران:** المقياس Recall@5 مكتوب لك كمثال. انسخ طريقته لـ Recall@1، ثم اكتب MRR.

✅ **سترى:** في اتجاه الصورة إلى النص تقريبا ٠٫٥٧ و٠٫٨٥ و٠٫٧٠، أي أفضل من التخمين بنحو مئة مرة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) recall_1 = (ranks == 1).mean()   — the same pattern as recall_5.
# 2) mrr = (1 / ranks).mean()
#
# خطوة ١: مثل سطر Recall@5 تماما، لكن قارن الرتب بالواحد.
# خطوة ٢: متوسط واحد على الرتب هو MRR.
# ────────────────────────────────────────────────────────────────────

def own_rank(scores):
    """Row i's right answer is column i. Its rank in that row: 1 = it came first."""
    ranks = []
    for i in range(len(scores)):
        beaten_by = (scores[i] > scores[i, i]).sum()        # wrong answers that scored higher
        ranks.append(beaten_by + 1)
    return np.array(ranks)


def retrieval_metrics(scores):
    ranks = own_rank(scores)
    recall_5 = (ranks <= 5).mean()           # example: how often the right answer is in the top 5
    # TODO: recall_1 = ... (how often the rank is exactly 1 — copy the line above)
    # مهمة: مثل السطر السابق، لكن الرتبة تساوي ١.
    # TODO: mrr = ... (the average of 1 / ranks)
    # مهمة: متوسط واحد على الرتب.
    return {"R@1": float(recall_1), "R@5": float(recall_5), "MRR": float(mrr)}


METRICS = pd.DataFrame([
    {"direction": "image -> text", **retrieval_metrics(SIMILARITY)},       # rows
    {"direction": "text -> image", **retrieval_metrics(SIMILARITY.T)},     # columns, turned into rows
])
print(METRICS.round(3).to_string(index=False))
print("\nrandom guessing: R@1 =", 1 / 200)

### Task 4 — Classify with zero labels

```
  5 class names ──► "a photo of a {c}" ──► embed_texts  ──►   5 vectors ──┐
                                                                         ├──► best match = the class
  400 photos    ─────────────────────────► embed_images ──► 400 vectors ──┘
```

No training at all — just the class names. Yesterday DINOv2 needed **320 labels** to reach 0.9875.
How close does CLIP get with **0**?

**Your job is two lines:** turn the five sentences into vectors, then pick the best class for
every photo.

✅ **You should see:** CLIP zero-shot ≈ 0.9975 — as good as yesterday's probe, with no labels.

<div dir="rtl" align="right">

### المهمة ٤ — صنف بلا تسميات

حول كل اسم فئة إلى جملة مثل «صورة لقطة»، ثم إلى متجه. وكل صورة تأخذ الفئة التي جملتها الأقرب إليها.
لا تدريب أبدا، أسماء الفئات فقط. أمس احتاج DINOv2 إلى **٣٢٠ تسمية**. فكم يصل CLIP بلا أي تسمية؟

**مهمتك سطران:** حول الجمل الخمس إلى متجهات، ثم اختر أفضل فئة لكل صورة.

✅ **سترى:** دقة CLIP بلا تدريب نحو ٠٫٩٩٧٥، مثل نتيجة الأمس، لكن بلا أي تسمية.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) sentence_vectors = embed_texts(sentences)
# 2) guesses = scores.argmax(axis=1)   — the column with the biggest score in every row
#
# خطوة ١: حول الجمل إلى متجهات بالدالة الجاهزة للنصوص.
# خطوة ٢: في كل صف، خذ موقع أكبر درجة.
# ────────────────────────────────────────────────────────────────────

CLASS_PHOTO_VECTORS = embed_images(CLASS_PATHS)              # 400 photos -> (400, 512), done once


def zero_shot(template):
    """Accuracy on the 400 class photos, using only the 5 class names."""
    sentences = [template.format(c=name) for name in CLASSES]          # "a photo of a cat", ...
    # TODO: sentence_vectors = ... (embed the 5 sentences with embed_texts)
    # مهمة: حول الجمل الخمس إلى متجهات.
    scores = CLASS_PHOTO_VECTORS @ sentence_vectors.T                  # (400 photos, 5 classes)
    # TODO: guesses = ... (for every photo, the position of its biggest score: .argmax(axis=1))
    # مهمة: لكل صورة، موقع أكبر درجة في صفها.
    return float((np.array(CLASSES)[guesses] == CLASS_LABELS).mean())  # how many guesses are right


ZERO_SHOT = zero_shot("a photo of a {c}")
print(pd.DataFrame([("CLIP zero-shot", 0, ZERO_SHOT),
                    ("DINOv2 + linear probe (D1)", 320, PROBE["dino_probe"]),
                    ("ResNet-18 fine-tune (W4D5)", 320, PROBE["w4d5_finetune"])],
                   columns=["method", "labels", "accuracy"]).to_string(index=False))

print("\nthe wording matters:")
for template in ["{c}", "a blurry photo of a {c}", "صورة {c}"]:
    print(f"  {zero_shot(template):.4f}  {template!r}")

### Task 5 — Break it

The last 8 captions are **true** for their own photo, but written to be hard for CLIP. For each one,
find where its own photo ranks among all 200. That is a **column** search, so use `SIMILARITY.T`.
Rank 1 = CLIP found it.

Why would it fail? CLIP learned which **words** go with which **pictures**. It never learned to
count, to tell left from right, or to read well — and it saw far less Arabic than English.

**Your job is one line:** rank every caption's own photo. The rest is given.

✅ **You should see:** "three cars" ranks 176, "four people" 117, and the Arabic caption 173 —
while the same sentence in English ranks 6.

<div dir="rtl" align="right">

### المهمة ٥ — اكسره

التعليقات الثمانية الأخيرة **صحيحة** عن صورها، لكنها كتبت لتكون صعبة على CLIP.
لكل تعليق، جد رتبة صورته بين الصور المئتين. هذا بحث في **عمود**، لذلك استعمل منقول الجدول.
الرتبة ١ تعني أن CLIP وجد الصورة.

لماذا يفشل؟ تعلم CLIP أي **الكلمات** تأتي مع أي **الصور**. لم يتعلم العد، ولا التمييز بين اليمين واليسار،
ولا القراءة الجيدة، ورأى من العربية أقل بكثير مما رأى من الإنجليزية.

**مهمتك سطر واحد:** جد رتبة صورة كل تعليق. والباقي جاهز.

✅ **سترى:** «ثلاث سيارات» في الرتبة ١٧٦، و«أربعة أشخاص» في ١١٧، والتعليق العربي في ١٧٣، بينما الجملة نفسها بالإنجليزية في الرتبة ٦.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) Columns are captions, so turn the table on its side: SIMILARITY.T
# 2) TEXT_RANKS = own_rank(SIMILARITY.T)
#
# خطوة ١: الأعمدة هي التعليقات، لذلك استعمل منقول الجدول.
# خطوة ٢: طبق دالة الرتبة من المهمة ٣ على منقول الجدول.
# ────────────────────────────────────────────────────────────────────

# TODO: TEXT_RANKS = own_rank(...)  (use SIMILARITY.T: for every caption, the rank of its own photo)
# مهمة: طبق دالة الرتبة على منقول الجدول.

HARD = CAPTIONS[CAPTIONS.hard_case.notna()].copy()           # the last 8 captions
HARD["rank_of_own_image"] = TEXT_RANKS[HARD.index]           # keep only their ranks
print(HARD[["hard_case", "caption", "rank_of_own_image"]].to_string(index=False))

ARABIC_RANK = int(HARD.loc[HARD.hard_case == "arabic", "rank_of_own_image"].iloc[0])
TWIN_RANK = int(HARD.loc[HARD.hard_case == "arabic_twin", "rank_of_own_image"].iloc[0])
print(f"\nArabic caption: its photo ranks {ARABIC_RANK}.  The same sentence in English: {TWIN_RANK}.")

In [ ]:
# === See the 8 hard cases with their ranks (1 = found) =================================
photos = [CAPTION_ROOT / "images" / f for f in HARD.file]
titles = [f"{kind}\nrank {rank}" for kind, rank in zip(HARD.hard_case, HARD.rank_of_own_image)]
show_images(photos[:4], titles[:4], "counting and left/right")
show_images(photos[4:], titles[4:], "left/right, a word in the photo, Arabic vs English")

### Takeaway

CLIP is a great **search engine** and a strong zero-shot classifier for everyday things. It is weak
exactly where a sentence needs **structure** — numbers, positions, written words — and on languages
it rarely saw while training.

<div dir="rtl" align="right">

### الخلاصة

نموذج CLIP **محرك بحث** ممتاز، ومصنف قوي بلا تدريب للأشياء اليومية.
لكنه ضعيف حين تحتاج الجملة إلى **بنية**: الأعداد، والمواقع، والكلمات المكتوبة، وفي اللغات التي نادرا ما رآها في التدريب.

</div>

## Section 3 — A VLM: ask the photo  *(~35 min)*

CLIP can only **score** a sentence you already wrote. A VLM **writes** the answer. It has three parts:

```
  photo ──► 1. vision encoder  (a ViT, like yesterday)  ──► patch vectors
                     │
                     ▼
            2. connector       squeezes them into a few IMAGE TOKENS
                     │
                     ▼
  [image tokens] + [your question] ──► 3. language model ──► the answer, one word at a time
```

To the language model, **a photo is just more tokens in the prompt**. More image tokens = more
detail, but a longer prompt and a slower answer.

Three tasks: open the model, ask it things, re-test it on the questions CLIP failed.

<div dir="rtl" align="right">

## القسم ٣ — نموذج الرؤية واللغة: اسأل الصورة  *(نحو ٣٥ دقيقة)*

نموذج CLIP **يقيم** فقط جملة كتبتها أنت. أما نموذج الرؤية واللغة **فيكتب** الجواب. وله ثلاثة أجزاء:

1. **مرمز الرؤية:** محول رؤية مثل محول الأمس، يحول الصورة إلى متجهات رقع.
2. **الموصل:** يضغط هذه المتجهات إلى عدد قليل من **رموز الصورة**.
3. **النموذج اللغوي:** يقرأ رموز الصورة ثم سؤالك، ويكتب الجواب كلمة بعد كلمة.

بالنسبة للنموذج اللغوي، **الصورة مجرد رموز إضافية في السؤال**. رموز أكثر تعني تفاصيل أكثر، لكن الجواب يصبح أبطأ.

</div>

### Task 1 — Open the VLM

The next cell loads SmolVLM (given). Then you count two things:

1. The **parameters** of each of its three parts: `vlm.model.vision_model`, `vlm.model.connector`
   and `vlm.model.text_model`.
2. The **image tokens** one photo costs. The processor can send a photo in two ways:
   - `LOW` — the whole photo as **one** 512 × 512 tile;
   - `HIGH` — the photo cut into a grid of tiles, plus one small overview.

   In `input_ids`, every position equal to `vlm.config.image_token_id` is one image token.

**Your job is two lines:** one inside each loop. `count_params(part)` is given.

✅ **You should see:** about 86 M + 12 M + 362 M parameters (most of the model is the language
model), and `{'LOW': 64, 'HIGH': 832}` image tokens.

<div dir="rtl" align="right">

### المهمة ١ — افتح نموذج الرؤية واللغة

الخلية التالية تحمل النموذج. ثم تعد أمرين:

1. **عدد المعاملات** في كل جزء من الأجزاء الثلاثة.
2. **عدد رموز الصورة** لصورة واحدة. يرسل المعالج الصورة بطريقتين:
   - الطريقة `LOW`: الصورة كلها في **بلاطة واحدة** بحجم ٥١٢ في ٥١٢.
   - الطريقة `HIGH`: الصورة مقسمة إلى شبكة بلاطات، مع نظرة عامة صغيرة.

   كل موقع في رموز الإدخال يساوي رقم رمز الصورة هو رمز صورة واحد.

**مهمتك سطران:** سطر داخل كل حلقة. ودالة عد المعاملات جاهزة.

✅ **سترى:** نحو ٨٦ مليونا و١٢ مليونا و٣٦٢ مليون معامل (معظم النموذج هو النموذج اللغوي)، و٦٤ رمز صورة مقابل ٨٣٢.

</div>

In [ ]:
# === Load SmolVLM (given) ==============================================================
from transformers import AutoModelForImageTextToText, AutoProcessor

VLM_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"
vlm = AutoModelForImageTextToText.from_pretrained(VLM_NAME, dtype=torch.float32).eval()
LOW = AutoProcessor.from_pretrained(VLM_NAME, do_image_splitting=False)    # the photo as one tile
HIGH = AutoProcessor.from_pretrained(VLM_NAME, do_image_splitting=True)    # the photo cut into tiles


def prepare(image, question, processor):
    """Put one photo and one question into the model's chat format."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    return processor(text=prompt, images=[image], return_tensors="pt")

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) PART_PARAMS[name] = count_params(part)
# 2) IMAGE_TOKENS[name] = int(is_image_token.sum())   — True counts as 1
#
# خطوة ١: نادي دالة عد المعاملات على الجزء، وضع النتيجة في القاموس.
# خطوة ٢: اجمع القيم الصحيحة في القناع، فكل قيمة صحيحة تساوي ١.
# ────────────────────────────────────────────────────────────────────

def count_params(part):
    """How many numbers (weights) are inside one part of the model."""
    return sum(p.numel() for p in part.parameters())


PARTS = {"vision encoder": vlm.model.vision_model,
         "connector": vlm.model.connector,
         "language model": vlm.model.text_model}
PART_PARAMS = {}
for name, part in PARTS.items():
    # TODO: PART_PARAMS[name] = ... (use count_params)
    # مهمة: ضع عدد معاملات هذا الجزء في القاموس.

photo = load_image(CAPTION_IMAGES[0])
IMAGE_TOKENS = {}
for name, processor in [("LOW", LOW), ("HIGH", HIGH)]:
    input_ids = prepare(photo, "Describe it.", processor)["input_ids"]
    is_image_token = input_ids == vlm.config.image_token_id          # True where the token is part of the photo
    # TODO: IMAGE_TOKENS[name] = ... (how many True values in is_image_token: int(... .sum()))
    # مهمة: عد القيم الصحيحة في القناع، وضع العدد في القاموس.

for name, n in PART_PARAMS.items():
    print(f"{name:<15} {n / 1e6:6.1f} M parameters")
print("\nimage tokens:", IMAGE_TOKENS)
print("one 512x512 tile = 32 x 32 = 1024 patches; the connector squeezes them into 64 tokens.")

### Task 2 — Ask it something

`ask(image, question)` works in three steps, and two of them are given:

1. `prepare(...)` the inputs (given);
2. `vlm.generate(...)` with `do_sample=False` — always pick the most likely next word, so you get
   the same answer every run (given);
3. the output **starts with your whole prompt again**:

```
   output = [ the prompt: prompt_length tokens ][ the new answer tokens ]
                                                 ▲
                                                 └── keep only this part: output[:, prompt_length:]
```

**Your job is one line:** step 3, keep only the new tokens. Then the cell captions three photos.

✅ **You should see:** three short, sensible sentences under the photos, about a second each.

<div dir="rtl" align="right">

### المهمة ٢ — اسأله شيئا

الدالة `ask` تعمل في ثلاث خطوات، وخطوتان منها جاهزتان:

1. تجهيز المدخلات (جاهزة).
2. توليد الجواب بدون عشوائية: يختار النموذج دائما الكلمة الأرجح، فتحصل على الجواب نفسه في كل مرة (جاهزة).
3. الخرج **يبدأ بالسؤال كله مرة أخرى**، ثم يأتي الجواب الجديد.

**مهمتك سطر واحد:** الخطوة ٣، احتفظ بالرموز **الجديدة** فقط، أي كل ما بعد طول السؤال. بعدها تكتب الخلية وصفا لثلاث صور.

✅ **سترى:** ثلاث جمل قصيرة ومعقولة تحت الصور، نحو ثانية لكل جملة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) A slice [:, n:] keeps every column from position n to the end.
# 2) new_tokens = output[:, prompt_length:]
#
# خطوة ١: القطع من الموقع n إلى النهاية يحتفظ بكل ما بعد n.
# خطوة ٢: اقطع الخرج من طول السؤال إلى النهاية.
# ────────────────────────────────────────────────────────────────────

def ask(image, question, processor=LOW, max_new_tokens=30):
    """The VLM's answer to one question about one photo."""
    inputs = prepare(image, question, processor)                        # 1. photo + question -> tokens
    output = vlm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # 2. write the answer
    prompt_length = inputs["input_ids"].shape[1]                        # how many tokens the prompt has
    # TODO: new_tokens = output[:, ...]  (3. keep only what comes after the prompt)
    # مهمة: احتفظ بالرموز التي بعد طول السؤال فقط.
    return processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()   # tokens -> text


photos, titles = [], []
for index in [3, 50, 120]:
    picture = load_image(CAPTION_IMAGES[index])
    started = time.perf_counter()
    answer = ask(picture, "Describe this image in one short sentence.")
    photos.append(picture)
    titles.append(textwrap.fill(answer, 34) + f"\n({time.perf_counter() - started:.1f} s)")
show_images(photos, titles)

### Task 3 — Re-test CLIP's failures

Ask the VLM about six of the photos CLIP got wrong — counting, left/right and the STOP sign — once
with `LOW` (64 image tokens) and once with `HIGH` (832). `is_right` marks every answer.

**Your job is one line:** inside the loop, ask the question. The loop, the timing and the marking
are given.

Then answer from your table: **does more detail help?** and **what does it cost?**

✅ **You should see:** `LOW` about 3 of 6 right, `HIGH` about 4 of 6, and `HIGH` several times
slower. The extra detail fixes "four people", but not "three cars" or the bus.

<div dir="rtl" align="right">

### المهمة ٣ — أعد اختبار أخطاء CLIP

اسأل نموذج الرؤية واللغة عن ست صور أخطأ فيها CLIP: العد، واليمين واليسار، ولافتة التوقف.
اسأل مرة بالطريقة `LOW` (٦٤ رمز صورة) ومرة بالطريقة `HIGH` (٨٣٢ رمزا). والدالة `is_right` تصحح كل جواب.

**مهمتك سطر واحد:** داخل الحلقة، اسأل السؤال. والحلقة وقياس الزمن والتصحيح كلها جاهزة.

ثم أجب من الجدول: **هل تساعد التفاصيل الإضافية؟** و**ما ثمنها؟**

✅ **سترى:** الطريقة المنخفضة تصيب نحو ٣ من ٦، والعالية نحو ٤ من ٦، والعالية أبطأ بعدة مرات. التفاصيل الإضافية تصلح «أربعة أشخاص»، لكنها لا تصلح «ثلاث سيارات» ولا الحافلة.

</div>

In [ ]:
# ────────────────────────────────────────────────────────────────────
# 1) ask(image, question, processor) — pass this loop's picture, question and processor.
# 2) answer = ask(picture, question, processor)
#
# خطوة ١: نادي دالة السؤال بصورة هذه الدورة وسؤالها ومعالجها.
# خطوة ٢: ضع ناتجها في متغير الجواب.
# ────────────────────────────────────────────────────────────────────

TESTS = [   # (photo file, question, a right answer contains one of these words)
    ("000000004551.jpg", "How many cars are in the picture? Answer with one number.", ["3", "three"]),
    ("000000020701.jpg", "How many zebras are in the picture? Answer with one number.", ["2", "two"]),
    ("000000000723.jpg", "How many people are in the picture? Answer with one number.", ["4", "four"]),
    ("000000005946.jpg", "Is the dog on the left or on the right of the person?", ["left"]),
    ("000000009999.jpg", "Is the bus on the left or on the right of the person?", ["left"]),
    ("000000009176.jpg", "What word is written on the sign?", ["stop"]),
]   # the truth comes from the captions in the dataset


def is_right(answer, right_words):
    answer = answer.lower()
    if "left" in answer and "right" in answer:      # "left or right" is not an answer
        return False
    return any(word in answer for word in right_words)


rows = []
for file, question, right_words in TESTS:
    picture = load_image(CAPTION_ROOT / "images" / file)
    for name, processor in [("LOW", LOW), ("HIGH", HIGH)]:
        started = time.perf_counter()
        # TODO: answer = ask(...)  (this picture, this question, this processor)
        # مهمة: اسأل النموذج عن هذه الصورة بهذا السؤال وهذا المعالج.
        rows.append((file, name, answer, is_right(answer, right_words), time.perf_counter() - started))

VLM_TABLE = pd.DataFrame(rows, columns=["file", "resolution", "answer", "correct", "seconds"])
print(VLM_TABLE.assign(answer=VLM_TABLE.answer.str[:50]).to_string(index=False))
SUMMARY = VLM_TABLE.groupby("resolution").agg(right=("correct", "sum"), seconds=("seconds", "mean"))
print("\n", SUMMARY.round(1))

### Write-up

In **one sentence each**: when would you use CLIP, and when a VLM? Think about the cost per photo,
about searching a million photos versus answering one question, and about what each one got wrong
today.

> *Example: "CLIP to search or sort many photos — one cheap vector each, compared in microseconds;
> a VLM when someone needs an answer about one photo, paying seconds per question, and more image
> tokens when the answer is in the small details."*

<div dir="rtl" align="right">

### الكتابة

في **جملة واحدة لكل منهما**: متى تستعمل CLIP، ومتى تستعمل نموذج رؤية ولغة؟
فكر في الكلفة لكل صورة، وفي البحث في مليون صورة مقابل الإجابة عن سؤال واحد، وفيما أخطأ فيه كل منهما اليوم.

> *مثال: «نموذج CLIP للبحث في صور كثيرة أو فرزها، لأن كل صورة متجه رخيص يقارن بسرعة كبيرة. ونموذج الرؤية واللغة حين يحتاج أحد جوابا عن صورة واحدة، بكلفة ثوان لكل سؤال، ومع رموز صورة أكثر حين يكون الجواب في التفاصيل الصغيرة.»*

</div>

In [ ]:
YOUR_ANSWER = """
(write your two sentences here)
"""

## Wrap-up

- **CLIP** = two encoders trained so that a photo and its caption land close together. Search,
  retrieval metrics and zero-shot classification are all read from **one similarity table**.
- It fails on counting, positions, words in photos, and languages it rarely saw.
- **A VLM** = vision encoder → connector → language model. A photo is just more tokens in the
  prompt — and more image tokens see more detail, at a price you measured.

Tomorrow (**D3**) the input changes again: sound. A waveform becomes a spectrogram, the spectrogram
becomes tokens, and a transformer writes down what was said.

<div dir="rtl" align="right">

## الخلاصة

- نموذج **CLIP** = مرمزان دربا لتقع الصورة وتعليقها قريبين من بعضهما. البحث ومقاييس الاسترجاع والتصنيف بلا تدريب كلها تقرأ من **جدول تشابه واحد**.
- ويفشل في العد، والمواقع، والكلمات داخل الصور، واللغات التي نادرا ما رآها.
- **نموذج الرؤية واللغة** = مرمز رؤية ← موصل ← نموذج لغوي. الصورة مجرد رموز إضافية في السؤال، ورموز أكثر ترى تفاصيل أكثر، بثمن قسته بنفسك.

وغدا (**اليوم ٣**) يتغير المدخل مرة أخرى: الصوت. تصبح الموجة صورة طيفية، وتصبح الصورة الطيفية رموزا، ويكتب المحول ما قيل.

</div>

In [ ]:
# === Checks: run this last =============================================================
check(IMAGE_VECTORS.shape == (200, 512) and TEXT_VECTORS.shape == (200, 512)
      and np.allclose(np.linalg.norm(IMAGE_VECTORS, axis=1), 1, atol=1e-5),
      f"photos and captions must be 512-number vectors of length 1 — got {IMAGE_VECTORS.shape}, {TEXT_VECTORS.shape}",
      f"يجب أن تكون متجهات الصور والتعليقات من ٥١٢ رقما وطولها ١ — والناتج {IMAGE_VECTORS.shape}، {TEXT_VECTORS.shape}")
check(list(top_k(np.array([0.1, 0.9, 0.5]), 2)) == [1, 2],
      "top_k must return the positions of the biggest scores, best first",
      "يجب أن تعيد الدالة مواقع أكبر الدرجات، الأفضل أولا")
check(all(METRICS["R@5"] >= METRICS["R@1"]) and METRICS["R@1"].min() > 0.3,
      f"Recall@5 must be at least Recall@1, and both directions far above chance (0.005) — got\n{METRICS}",
      f"يجب ألا يقل Recall@5 عن Recall@1، وأن يكون الاتجاهان أعلى بكثير من الصدفة — والناتج\n{METRICS}")
check(ZERO_SHOT > 0.9,
      f"zero-shot CLIP must classify the five everyday classes above 0.9 with no labels — got {ZERO_SHOT:.4f}",
      f"يجب أن يتجاوز تصنيف CLIP بلا تدريب ٠٫٩ — والناتج {ZERO_SHOT:.4f}")
check(ARABIC_RANK > TWIN_RANK,
      f"the Arabic caption must rank its photo worse than its English twin does — got {ARABIC_RANK} vs {TWIN_RANK}",
      f"يجب أن تكون رتبة صورة التعليق العربي أسوأ من رتبة توأمه الإنجليزي — والناتج {ARABIC_RANK} مقابل {TWIN_RANK}")
check(IMAGE_TOKENS["LOW"] < IMAGE_TOKENS["HIGH"] and len(VLM_TABLE) == 12,
      f"HIGH must use more image tokens than LOW, and all 6 questions must be asked both ways — got {IMAGE_TOKENS}, {len(VLM_TABLE)} rows",
      f"يجب أن تستعمل الطريقة العالية رموزا أكثر، وأن تسأل الأسئلة الستة بالطريقتين — والناتج {IMAGE_TOKENS}، {len(VLM_TABLE)} صفا")
report()